# Titanic 28: learning_rate + Early Stopping 5-Fold CV

`titanic_15.ipynb`에서 실제 `submission/titanic_result_15.csv`를 만든 모델을 기준으로 한 독립 하이퍼파라미터 실험이다.
기준 피처는 `GenderClass`, `GenderIsChild`, `ClassIsChild`, `AgeBand`이며, 전처리/결측치/인코딩/holdout split/평가/제출 형식은 Titanic 15와 동일하게 유지한다.

선택에는 train 데이터의 5-Fold CV ROC-AUC만 사용한다. test는 최종 예측에만 사용한다.

In [1]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier
from common.interaction_experiments import FeaturePreprocessor
from common.feature_experiments import baseline_parameters

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

train = pd.read_csv('csv/train.csv')
test = pd.read_csv('csv/test.csv')
submission_template = pd.read_csv('csv/submission.csv')

target_col = 'survived'
id_col = 'passengerid'
BASE_FEATURES = ['GenderClass', 'GenderIsChild', 'ClassIsChild', 'AgeBand']
X = train.drop(columns=[target_col, id_col])
y = train[target_col].astype('int8')
X_test_raw = test.drop(columns=[id_col])

assert X.columns.equals(X_test_raw.columns)
assert train[id_col].is_unique and test[id_col].is_unique
assert set(train[id_col]).isdisjoint(test[id_col])

## Random State Audit

Python/NumPy 전역 seed를 Kernel Restart 후 처음부터 고정한다. 모든 데이터 분할과 탐색 샘플링은 명시적인 `random_state=42`, CatBoost는 Titanic 15의 `random_state=42`(CatBoost 내부 `random_seed=42`와 같은 alias)를 사용한다.

In [2]:
RANDOM_AUDIT = pd.DataFrame([
    ('Python random', 42, 'random.seed(42)'),
    ('NumPy', 42, 'np.random.seed(42)'),
    ('train_test_split', 42, 'explicit random_state'),
    ('StratifiedKFold', 42, 'shuffle=True, explicit random_state'),
    ('CatBoost', 42, 'baseline random_state alias -> random_seed'),
    ('RandomizedSearch', 'N/A', 'not used in Titanic 28'),
], columns=['Component', 'Random State', 'Code basis'])
display(RANDOM_AUDIT)
assert RANDOM_AUDIT.loc[RANDOM_AUDIT['Random State'].ne('N/A'), 'Random State'].eq(42).all()
print('모든 사용 stochastic component의 seed가 코드에서 42로 고정되었습니다.')

,Component,Random State,Code basis
0,Python random,42,random.seed(42)
1,NumPy,42,np.random.seed(42)
2,train_test_split,42,explicit random_state
3,StratifiedKFold,42,"shuffle=True, explicit random_state"
4,CatBoost,42,baseline random_state alias -> random_seed
5,RandomizedSearch,N/A,not used in Titanic 28


모든 사용 stochastic component의 seed가 코드에서 42로 고정되었습니다.


## 고정된 데이터 분할과 누수 방지 구조

Titanic 15의 기본 25% stratified holdout을 같은 seed로 재현한다. CV에서는 매 fold마다 새 전처리기를 만들고 Train Fold에만 `fit_transform`, Validation Fold에는 `transform`만 적용한다.

In [3]:
train_part, valid_part = train_test_split(
    train, test_size=0.25, stratify=train[target_col], random_state=SEED
)
X_tr_raw = train_part.drop(columns=[target_col, id_col])
y_tr = train_part[target_col].astype('int8')
X_valid_raw = valid_part.drop(columns=[target_col, id_col])
y_valid = valid_part[target_col].astype('int8')
assert len(train_part) == 687 and len(valid_part) == 229

BASE_PARAMS = baseline_parameters()
assert BASE_PARAMS['random_state'] == SEED
print('Titanic 15 명시 파라미터:', BASE_PARAMS)
print('고정 피처:', BASE_FEATURES)

Titanic 15 명시 파라미터: {'verbose': 0, 'random_state': 42, 'cat_features': [], 'allow_writing_files': False}
고정 피처: ['GenderClass', 'GenderIsChild', 'ClassIsChild', 'AgeBand']


In [4]:
def prepare_fold(X_train_raw, X_valid_raw):
    prep = FeaturePreprocessor(BASE_FEATURES)
    X_train_model = prep.fit_transform(X_train_raw)
    X_valid_model = prep.transform(X_valid_raw)
    assert X_train_model.columns.equals(X_valid_model.columns)
    assert np.isfinite(X_train_model.to_numpy()).all()
    assert np.isfinite(X_valid_model.to_numpy()).all()
    return prep, X_train_model, X_valid_model


def fit_model(X_train_model, y_train, overrides=None, **fit_kwargs):
    params = BASE_PARAMS | (overrides or {})
    model = CatBoostClassifier(**params)
    model.fit(X_train_model, y_train, **fit_kwargs)
    return model


def auc_scores(model, X_train_model, y_train, X_valid_model, y_valid):
    positive_index = list(model.classes_).index(1)
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train_model)[:, positive_index])
    valid_auc = roc_auc_score(y_valid, model.predict_proba(X_valid_model)[:, positive_index])
    return train_auc, valid_auc, train_auc - valid_auc


def baseline_cv():
    rows = []
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), 1):
        _, X_fold_train, X_fold_valid = prepare_fold(X.iloc[tr_idx], X.iloc[va_idx])
        model = fit_model(X_fold_train, y.iloc[tr_idx])
        train_auc, valid_auc, gap = auc_scores(
            model, X_fold_train, y.iloc[tr_idx], X_fold_valid, y.iloc[va_idx]
        )
        rows.append({'fold': fold, 'train_auc': train_auc,
                     'validation_auc': valid_auc, 'gap': gap})
    folds = pd.DataFrame(rows)
    summary = {
        'cv_mean': folds['validation_auc'].mean(),
        'cv_std': folds['validation_auc'].std(ddof=1),
        'train_auc': folds['train_auc'].mean(),
        'gap': folds['gap'].mean(),
    }
    return folds, summary


baseline_folds, baseline_cv_summary = baseline_cv()
display(baseline_folds)
print('Titanic 15 actual-submission baseline CV:', baseline_cv_summary)

,fold,train_auc,validation_auc,gap
0,1,0.957912,0.906140,0.051772
1,2,0.958143,0.924358,0.033785
2,3,0.963202,0.886728,0.076475
3,4,0.956995,0.905861,0.051135
4,5,0.957728,0.914505,0.043222


Titanic 15 actual-submission baseline CV: {'cv_mean': np.float64(0.9075184337655735), 'cv_std': np.float64(0.013868055749294898), 'train_auc': np.float64(0.9587961681724849), 'gap': np.float64(0.05127773440691141)}


## learning_rate별 Early Stopping CV

각 fold의 validation fold만 `eval_set`으로 사용한다. `iterations=3000`, `early_stopping_rounds=150`, `loss_function='Logloss'`, `eval_metric='AUC'`로 Best Iteration을 찾는다.

In [5]:
learning_rates = [0.01, 0.015, 0.02, 0.03, 0.05]
candidate_rows = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for learning_rate in learning_rates:
    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), 1):
        _, X_fold_train, X_fold_valid = prepare_fold(X.iloc[tr_idx], X.iloc[va_idx])
        overrides = {'learning_rate': learning_rate, 'iterations': 3000,
                     'loss_function': 'Logloss', 'eval_metric': 'AUC'}
        model = fit_model(
            X_fold_train, y.iloc[tr_idx], overrides,
            eval_set=(X_fold_valid, y.iloc[va_idx]),
            early_stopping_rounds=150, use_best_model=True, verbose=False,
        )
        train_auc, valid_auc, gap = auc_scores(
            model, X_fold_train, y.iloc[tr_idx], X_fold_valid, y.iloc[va_idx]
        )
        best_iteration = int(model.get_best_iteration()) + 1
        assert best_iteration >= 1
        candidate_rows.append({'learning_rate': learning_rate, 'fold': fold,
                               'best_iteration': best_iteration, 'train_auc': train_auc,
                               'validation_auc': valid_auc, 'gap': gap})
    print('완료: learning_rate', learning_rate)

cv_folds = pd.DataFrame(candidate_rows)
cv_summary = (cv_folds.groupby('learning_rate', as_index=False)
              .agg(cv_mean=('validation_auc', 'mean'),
                   cv_std=('validation_auc', 'std'),
                   mean_best_iteration=('best_iteration', 'mean'),
                   median_best_iteration=('best_iteration', 'median'),
                   train_auc=('train_auc', 'mean'), gap=('gap', 'mean'))
              .sort_values(['cv_mean', 'cv_std'], ascending=[False, True], ignore_index=True))
display(cv_folds)
display(cv_summary)
best = cv_summary.iloc[0]
BEST_LEARNING_RATE = float(best['learning_rate'])
FINAL_ITERATIONS = max(1, int(round(best['median_best_iteration'])))
print('Best learning_rate:', BEST_LEARNING_RATE)
print('CV mean/median Best Iteration:', best['mean_best_iteration'], best['median_best_iteration'])
print('Final iterations (rounded CV median):', FINAL_ITERATIONS)

완료: learning_rate 0.01


완료: learning_rate 0.015


완료: learning_rate 0.02


완료: learning_rate 0.03


완료: learning_rate 0.05


,learning_rate,fold,best_iteration,train_auc,validation_auc,gap
0,0.010,1,42,0.923003,0.904699,0.018304
1,0.010,2,182,0.928586,0.929189,-0.000603
2,0.010,3,6,0.922798,0.893974,0.028824
3,0.010,4,2,0.909854,0.919209,-0.009355
4,0.010,5,22,0.921904,0.911200,0.010704
5,0.015,1,41,0.925307,0.906140,0.019167
6,0.015,2,126,0.929255,0.929761,-0.000506
7,0.015,3,9,0.923645,0.897343,0.026302
8,0.015,4,2,0.909696,0.919209,-0.009513
9,0.015,5,19,0.923143,0.914760,0.008383


,learning_rate,cv_mean,cv_std,mean_best_iteration,median_best_iteration,train_auc,gap
0,0.030,0.914325,0.012962,50.2,20.0,0.925627,0.011302
1,0.020,0.913796,0.013861,20.4,17.0,0.920217,0.006421
2,0.050,0.913636,0.013260,18.0,20.0,0.923074,0.009438
3,0.015,0.913443,0.012387,39.4,19.0,0.922209,0.008767
4,0.010,0.911654,0.013470,50.8,22.0,0.921229,0.009575


Best learning_rate: 0.03
CV mean/median Best Iteration: 50.2 20.0
Final iterations (rounded CV median): 20


In [6]:
_, X_holdout_train, X_holdout_valid = prepare_fold(X_tr_raw, X_valid_raw)
baseline_holdout_model = fit_model(X_holdout_train, y_tr)
baseline_holdout = auc_scores(baseline_holdout_model, X_holdout_train, y_tr,
                              X_holdout_valid, y_valid)
best_holdout_params = {'learning_rate': BEST_LEARNING_RATE, 'iterations': 3000,
                       'loss_function': 'Logloss', 'eval_metric': 'AUC'}
best_holdout_model = fit_model(
    X_holdout_train, y_tr, best_holdout_params,
    eval_set=(X_holdout_valid, y_valid), early_stopping_rounds=150,
    use_best_model=True, verbose=False,
)
best_holdout = auc_scores(best_holdout_model, X_holdout_train, y_tr,
                          X_holdout_valid, y_valid)
holdout_best_iteration = int(best_holdout_model.get_best_iteration()) + 1

comparison = pd.DataFrame([
    {'Model': 'Baseline Titanic 15', 'Parameter': str(BASE_PARAMS),
     'CV Mean': baseline_cv_summary['cv_mean'], 'CV Std': baseline_cv_summary['cv_std'],
     'Validation AUC': baseline_holdout[1], 'Gap': baseline_holdout[2]},
    {'Model': 'Titanic 28 Best',
     'Parameter': str({'learning_rate': BEST_LEARNING_RATE,
                       'final_iterations': FINAL_ITERATIONS,
                       'early_stopping_rounds': 150}),
     'CV Mean': best['cv_mean'], 'CV Std': best['cv_std'],
     'Validation AUC': best_holdout[1], 'Gap': best_holdout[2]},
])
display(comparison)
print('Best learning_rate:', BEST_LEARNING_RATE)
print('Holdout Best Iteration:', holdout_best_iteration)
print('Final iterations:', FINAL_ITERATIONS)
print('CV Mean:', best['cv_mean'], 'CV Std:', best['cv_std'])
FINAL_PARAMS = BASE_PARAMS | {
    'learning_rate': BEST_LEARNING_RATE,
    'iterations': FINAL_ITERATIONS,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
}

,Model,Parameter,CV Mean,CV Std,Validation AUC,Gap
0,Baseline Titanic 15,"{'verbose': 0, 'random_state': 42, 'cat_featur...",0.907518,0.013868,0.900024,0.062311
1,Titanic 28 Best,"{'learning_rate': 0.03, 'final_iterations': 20...",0.914325,0.012962,0.906367,0.019120


Best learning_rate: 0.03
Holdout Best Iteration: 24
Final iterations: 20
CV Mean: 0.914324942791762 CV Std: 0.012961501591864434


## 최종 모델과 제출 파일

선택된 파라미터로 전체 train에 전처리를 fit하고 test에는 transform만 적용한다. test는 CV, 파라미터 선택, Early Stopping에 사용하지 않는다.

In [7]:
final_prep = FeaturePreprocessor(BASE_FEATURES)
X_full_model = final_prep.fit_transform(X)
X_test_model = final_prep.transform(X_test_raw)
assert X_full_model.columns.equals(X_test_model.columns)
final_model = CatBoostClassifier(**FINAL_PARAMS)
final_model.fit(X_full_model, y)
positive_index = list(final_model.classes_).index(1)
predictions = final_model.predict_proba(X_test_model)[:, positive_index]

# Titanic 15와 같은 template/id mapping 방식으로 생성한다.
assert submission_template.columns.tolist() == [id_col, target_col]
assert len(submission_template) == len(test)
assert submission_template[id_col].is_unique and submission_template[id_col].notna().all()
assert set(submission_template[id_col]) == set(test[id_col])
result = submission_template.copy(deep=True)
by_id = pd.Series(predictions, index=test[id_col].to_numpy())
result[target_col] = result[id_col].map(by_id)

# 저장 직전 검증: 행 수, ID/순서, 확률 dtype, NaN, 범위.
assert len(result) == len(test) == len(submission_template)
pd.testing.assert_series_equal(result[id_col], submission_template[id_col], check_names=True)
assert pd.api.types.is_float_dtype(result[target_col])
assert result[target_col].notna().all()
assert np.isfinite(result[target_col].to_numpy()).all()
assert result[target_col].between(0.0, 1.0).all()

output_path = Path('titanic_28_result.csv')
result.to_csv(output_path, index=False)
saved = pd.read_csv(output_path)
assert saved.shape == submission_template.shape == (len(test), 2)
assert saved.columns.equals(submission_template.columns)
pd.testing.assert_frame_equal(saved.drop(columns=target_col), submission_template.drop(columns=target_col))
assert pd.api.types.is_float_dtype(saved[target_col])
assert saved[target_col].notna().all() and saved[target_col].between(0.0, 1.0).all()
np.testing.assert_allclose(saved[target_col], by_id.loc[submission_template[id_col]], rtol=1e-12, atol=1e-15)
print('Submission 검증 통과:', output_path, saved.shape)

Submission 검증 통과: titanic_28_result.csv (393, 2)
